In [1]:
# 1. Install all required libraries
!pip install -q ultralytics roboflow huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 132.5 MB/s eta 0:00:00


In [2]:
# 2. Import libraries
from roboflow import Roboflow
from ultralytics import YOLO
from huggingface_hub import notebook_login
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# 3. Log in to Roboflow
# Go to app.roboflow.com/settings/api to get your API key
# It's a "private" key, so it won't be shown in the uploaded file.
rf = Roboflow(api_key="#############")

In [4]:
# 4. Download the CarDD dataset (YOLOv8 format)
# This is the magic step. It will download, unzip, and set up everything.
print("Downloading CarDD dataset from Roboflow (this is much faster)...")
# We specify the workspace, project, and version.
project = rf.workspace("ratchakrit").project("car-damage-merged-qefuo")
dataset = project.version(1).download("yolov8")
print("Dataset download complete.")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to car-damage-merged-1 in yolov8:: 100%|██████████| 17926/17926 [00:03<00:00, 5893.34it/s]

Dataset download complete.


In [ ]:
# --- ADD THIS NEW CODE BLOCK AFTER DOWNLOADING ---

import yaml

# This is the path to your dataset from the previous step
dataset_location = dataset.location
data_yaml_path = os.path.join(dataset_location, 'data.yaml')

print(f"Original data.yaml path: {data_yaml_path}")

# Define the *correct* class names from your screenshot
# We need to find out the order. Let's assume the Roboflow
# download page listed them alphabetically or by ID.
# For now, we will use the names from your screenshot.
# The order MUST match the original file. Let's load it to check.

with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print(f"Original names: {data['names']}")
# This will probably print ['1', '2', '3', '4', '5', '2 0 0 0 1 1 1 1 0 0 0']
# We will replace them in the order from the screenshot.

data['names'] = [
    'crack',
    'dent',
    'glass_shatter',
    'lamp_broken',
    'scratch',
    'tire_flat'
]

# Write the new, correct data back to the file
with open(data_yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"Updated data.yaml with correct names: {data['names']}")

Original data.yaml path: /content/car-damage-merged-1/data.yaml
Original names: ['1', '2', '2 0 0 0 1 1 1 1 0 0 0', '3', '4', '5']
Updated data.yaml with correct names: ['crack', 'dent', 'glass_shatter', 'lamp_broken', 'scratch', 'tire_flat']


In [5]:
# 5. Train a BIGGER, SMARTER model
print("Starting YOLOv8 training with a nano model (yolov8n.pt)...")
from ultralytics import YOLO
import os

# We are loading the 'nano' model. (faster for a sprint, only have 48 hrs)
model = YOLO('yolov8n.pt')
# --- --- --- --- --- --- --- ---

# Train it
results = model.train(
    data=os.path.join(dataset.location, 'data.yaml'),
    epochs=30,
    imgsz=640,
    patience=5,         # Stop early if it doesn't improve
    close_mosaic=4,     # *** THIS IS THE KEY ***
                         # It turns off mosaic for the last 4 epochs
                         # This will improve small object accuracy.
    name='yolov8_nano_sprint'
)

print("Training complete! Should see great mAP scores.")

Starting YOLOv8 training with a nano model (yolov8n.pt)...
Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=4, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/car-damage-merged-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_nano_sprint, nbs=64, nms=False, opset=None, optimize=False, opt

In [6]:
# 6. Log in to Hugging Face
print("Logging in to Hugging Face...")
# You'll need a "write" token from hf.co/settings/tokens
notebook_login()

Logging in to Hugging Face...


In [10]:
# Complete Model Upload Script - Ready to Run!
print("=" * 70)
print("UPLOADING MODEL TO HUGGING FACE HUB")
print("=" * 70)

import os
import tempfile
import shutil
from pathlib import Path
from huggingface_hub import HfApi, create_repo, login

# Your model path
MODEL_PATH = '/content/runs/detect/yolov8_nano_sprint/weights/best.pt'

# Your Hugging Face repository name
YOUR_MODEL_NAME = "CharbelMsalem/yolov8n-finetuned-datamatics-damage"

# Step 1: Authenticate with Hugging Face
print("\n🔐 Step 1: Authenticating with Hugging Face...")
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("   ✅ Authenticated successfully!")
except Exception as e:
    print("   ⚠️  Authentication note:", e)
    print("   Continuing without authentication (may work for public repos)")

# Step 2: Check if model exists
print(f"\n📁 Step 2: Checking model file...")
model_file = Path(MODEL_PATH)
if not model_file.exists():
    print(f"   ❌ ERROR: Model not found at {MODEL_PATH}")
    print(f"\n   Available models:")
    runs_dir = Path('/content/runs/detect')
    if runs_dir.exists():
        for run in sorted(runs_dir.glob('yolov8*'), key=os.path.getmtime, reverse=True):
            weights = run / 'weights' / 'best.pt'
            if weights.exists():
                print(f"      ✅ {weights}")
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")

print(f"   ✅ Model found!")
print(f"   📊 Size: {model_file.stat().st_size / (1024*1024):.2f} MB")

# Step 3: Create temporary directory with model
print(f"\n📦 Step 3: Preparing upload package...")
temp_dir = tempfile.mkdtemp()
temp_model_path = Path(temp_dir) / 'best.pt'

try:
    # Copy model to temp directory
    shutil.copy(model_file, temp_model_path)
    print(f"   ✅ Model copied to temporary directory")

    # Step 4: Create repository on Hugging Face
    print(f"\n🏗️  Step 4: Creating/verifying repository...")
    api = HfApi()

    try:
        create_repo(
            repo_id=YOUR_MODEL_NAME,
            repo_type="model",
            exist_ok=True
        )
        print(f"   ✅ Repository ready: {YOUR_MODEL_NAME}")
    except Exception as e:
        print(f"   ℹ️  Repository note: {e}")

    # Step 5: Upload model
    print(f"\n🚀 Step 5: Uploading model to Hugging Face Hub...")
    print(f"   This may take a moment...")

    result = api.upload_folder(
        folder_path=temp_dir,
        repo_id=YOUR_MODEL_NAME,
        repo_type="model",
    )

    # Step 6: Success!
    print(f"\n" + "=" * 70)
    print("✅ SUCCESS! MODEL UPLOADED TO HUGGING FACE!")
    print("=" * 70)
    print(f"\n🎉 Your model is now available at:")
    print(f"   https://huggingface.co/{YOUR_MODEL_NAME}")
    print(f"\n📝 To use your model in Python:")
    print(f"   from ultralytics import YOLO")
    print(f"   model = YOLO('hf://{YOUR_MODEL_NAME}')")
    print(f"\n📝 Or download directly:")
    print(f"   from huggingface_hub import hf_hub_download")
    print(f"   model_path = hf_hub_download(")
    print(f"       repo_id='{YOUR_MODEL_NAME}',")
    print(f"       filename='best.pt'")
    print(f"   )")
    print("=" * 70)

except Exception as e:
    print(f"\n❌ ERROR during upload:")
    print(f"   {type(e).__name__}: {e}")
    print(f"\n💡 Troubleshooting:")
    print(f"   1. Make sure you're logged into Hugging Face")
    print(f"   2. Check your internet connection")
    print(f"   3. Verify the model file exists")

finally:
    # Step 7: Cleanup
    print(f"\n🧹 Cleaning up temporary files...")
    shutil.rmtree(temp_dir, ignore_errors=True)
    print(f"   ✅ Cleanup complete!")

print("\n✨ Script finished!")

UPLOADING MODEL TO HUGGING FACE HUB

🔐 Step 1: Authenticating with Hugging Face...
   ⚠️  Authentication note: Secret HF_TOKEN does not exist.
   Continuing without authentication (may work for public repos)

📁 Step 2: Checking model file...
   ✅ Model found!
   📊 Size: 5.96 MB

📦 Step 3: Preparing upload package...
   ✅ Model copied to temporary directory

🏗️  Step 4: Creating/verifying repository...
   ✅ Repository ready: CharbelMsalem/yolov8n-finetuned-datamatics-damage

🚀 Step 5: Uploading model to Hugging Face Hub...
   This may take a moment...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpps15szsf/best.pt    :   9%|8         |  534kB / 6.25MB            


✅ SUCCESS! MODEL UPLOADED TO HUGGING FACE!

🎉 Your model is now available at:
   https://huggingface.co/CharbelMsalem/yolov8n-finetuned-datamatics-damage

📝 To use your model in Python:
   from ultralytics import YOLO
   model = YOLO('hf://CharbelMsalem/yolov8n-finetuned-datamatics-damage')

📝 Or download directly:
   from huggingface_hub import hf_hub_download
   model_path = hf_hub_download(
       repo_id='CharbelMsalem/yolov8n-finetuned-datamatics-damage',
       filename='best.pt'
   )

🧹 Cleaning up temporary files...
   ✅ Cleanup complete!

✨ Script finished!
